# 02 · Train, evaluate and explain (Kaggle / Colab GPU)

**Kaggle setup**
1. *Add data*: search `nih-chest-xrays/data` (mounted at `/kaggle/input/data`).
2. Upload this project folder (zip of `src/`, `configs/`, `app/`) as a private Kaggle dataset and add it too,
   or `git clone` your repo in the first cell.
3. *Settings → Accelerator → GPU T4 x2 or P100*, and turn *Internet* on (for ImageNet weights).

Everything here calls the modules in `src/`. This notebook just runs experiments and shows results.

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path

CODE_SRC = Path('/kaggle/input/cxr-capstone-code')    # <- name of your uploaded code dataset
PROJECT = Path('/kaggle/working/project')
if CODE_SRC.exists() and not PROJECT.exists():
    shutil.copytree(CODE_SRC, PROJECT)
os.chdir(PROJECT); sys.path.insert(0, str(PROJECT))

RAW_DATA = '/kaggle/input/data'
FAST_DATA = '/kaggle/temp/nih256'   # resized copy, built below. /kaggle/temp is NOT saved as output,
                                     # so the 5 GB of images don't bloat your saved notebook version
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

### 1. Pre-resize images to 256px once (~20-30 min, makes every epoch much faster)

In [ ]:
!python -m src.preprocess --src {RAW_DATA} --dst {FAST_DATA} --size 256 --workers 4

### 2. Smoke test (500 images, 1 epoch) to catch errors before the long runs

In [ ]:
!python -m src.train --config configs/default.yaml --data-dir {FAST_DATA} --out-dir outputs/smoke --subset 500 --epochs 1

### 3. Experiments: how to handle class imbalance
Same backbone (DenseNet-121), three losses. Each run takes roughly 1 to 2 h on a T4.
If you are short on GPU time, run `weighted_bce` first; it is the main model.

In [ ]:
EXPERIMENTS = {
    'densenet121_bce':   ['--loss', 'bce'],
    'densenet121_wbce':  ['--loss', 'weighted_bce'],
    'densenet121_focal': ['--loss', 'focal'],
    # 'efficientnet_b0_wbce': ['--loss', 'weighted_bce', '--backbone', 'efficientnet_b0'],
}
for name, extra in EXPERIMENTS.items():
    if Path(f'outputs/{name}/best_model.pt').exists():
        print('skip', name); continue
    subprocess.run([sys.executable, '-m', 'src.train', '--config', 'configs/default.yaml', '--data-dir', FAST_DATA,
                    '--out-dir', f'outputs/{name}', '--num-workers', '4', *extra], check=True)

### 4. Evaluate on the official test split (thresholds tuned on validation)

In [ ]:
for name in EXPERIMENTS:
    subprocess.run([sys.executable, '-m', 'src.evaluate', '--checkpoint', f'outputs/{name}/best_model.pt',
                    '--data-dir', FAST_DATA, '--num-workers', '4'], check=True)

In [ ]:
import pandas as pd
rows = {}
for name in EXPERIMENTS:
    m = json.loads(Path(f'outputs/{name}/metrics_test.json').read_text())
    rows[name] = {**{f'AUROC {c}': v['auroc'] for c, v in m['per_class'].items()}, **m['summary']}
comparison = pd.DataFrame(rows).T
comparison[['mean_auroc', 'mean_auprc', 'mean_f1']].round(4)

In [ ]:
comparison.filter(like='AUROC').T.round(3)

In [ ]:
from IPython.display import Image as Show
BEST = 'densenet121_wbce'
display(Show(f'outputs/{BEST}/roc_curves.png')); display(Show(f'outputs/{BEST}/pr_curves.png'))

### 5. Is the model looking at the right place? Grad-CAM vs radiologist boxes
Uses the **original** 1024px images for the sharpest overlays. The script also checks that every box image is in the held-out test split.
Compare `energy_in_box` and `pointing_hit` with `box_fraction` (what a random heatmap would score).

In [ ]:
for method in ['gradcam', 'gradcam++']:
    subprocess.run([sys.executable, '-m', 'src.localization', '--checkpoint', f'outputs/{BEST}/best_model.pt',
                    '--data-dir', RAW_DATA, '--method', method], check=True)
pd.read_csv(f'outputs/{BEST}/localization_gradcam/summary.csv', index_col=0).round(3)

In [ ]:
# Worst cases: heatmap energy mostly outside the box. Look for shortcuts (text markers, tubes, image borders).
import glob, matplotlib.pyplot as plt
from PIL import Image
for tag in ['worst', 'best']:
    files = sorted(glob.glob(f'outputs/{BEST}/localization_gradcam/{tag}_*.png'))[:6]
    fig, axes = plt.subplots(1, len(files), figsize=(4 * len(files), 4))
    for ax, f in zip(axes, files): ax.imshow(Image.open(f)); ax.set_title(Path(f).stem[:40], fontsize=8); ax.axis('off')
    fig.suptitle(tag.upper()); plt.show()

### 6. Save outputs and optionally try the demo here

In [ ]:
# Download outputs/<BEST>/best_model.pt and thresholds.json from the Output tab to run the demo locally.
!zip -qr /kaggle/working/results.zip outputs -x '*last_model.pt'
# !pip -q install gradio && python app/app.py --checkpoint outputs/{BEST}/best_model.pt --share